# Speech Commands (TFDS) — load, explore, preprocess, train

Dataset: `speech_commands` v0.0.3 from TensorFlow Datasets
- 1-second clips, 16 kHz, int16 audio
- 12 classes: 10 keywords + `_silence_` + `_unknown_`
- Splits: train 85,511 · validation 10,102 · test 4,890
- **Download ≈ 2.4 GB, needs ≈ 8.2 GB disk** (first run only; it's cached afterwards)

Run the cells top to bottom. Select a Python 3.9–3.12 kernel in VS Code (top-right of the notebook).

## 1. Install packages (run once)

In [ ]:
%pip install -q tensorflow tensorflow-datasets matplotlib numpy

## 2. Imports and settings

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from IPython.display import Audio, display

print("TensorFlow:", tf.__version__)
print("TFDS:", tfds.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

# Where the dataset is downloaded/cached. Change if your C: drive is short on space.
DATA_DIR = os.path.join(os.path.expanduser("~"), "tensorflow_datasets")

SAMPLE_RATE = 16000
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

## 3. Download + load the dataset
First run downloads ~2.4 GB and prepares it — this can take 15–40 min depending on your internet.

In [ ]:
(train_ds, val_ds, test_ds), info = tfds.load(
    'speech_commands',
    split=['train', 'validation', 'test'],
    as_supervised=True,      # gives (audio, label) tuples
    with_info=True,
    data_dir=DATA_DIR,
)

LABELS = info.features['label'].names
NUM_CLASSES = len(LABELS)
print("Classes:", LABELS)
print({s: info.splits[s].num_examples for s in info.splits})
info

## 4. Look at a few samples (waveform + listen)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 8))
samples = []
for ax, (audio, label) in zip(axes.flat, train_ds.take(9)):
    a = audio.numpy()
    samples.append((a, label.numpy()))
    ax.plot(a)
    ax.set_title(f"{LABELS[label]}  ({len(a)} samples)")
    ax.set_yticks([])
plt.tight_layout()
plt.show()

# Play the first clip
a, l = samples[0]
print("Label:", LABELS[l])
display(Audio(a, rate=SAMPLE_RATE))

## 5. Class distribution (optional — iterates the whole train split, ~1–2 min)

In [ ]:
counts = np.zeros(NUM_CLASSES, dtype=int)
for _, label in train_ds.map(lambda a, l: (0, l)).batch(4096):
    counts += np.bincount(label.numpy(), minlength=NUM_CLASSES)

plt.figure(figsize=(10, 4))
plt.bar(LABELS, counts)
plt.xticks(rotation=45)
plt.title("Train class counts (note: _unknown_ dominates)")
plt.show()
dict(zip(LABELS, counts))

## 6. Preprocessing: int16 → float, pad/trim to 1 s, log-mel spectrogram

In [ ]:
def to_float_fixed_length(audio, label):
    audio = tf.cast(audio, tf.float32) / 32768.0          # int16 -> [-1, 1]
    audio = audio[:SAMPLE_RATE]                            # trim
    pad = SAMPLE_RATE - tf.shape(audio)[0]
    audio = tf.pad(audio, [[0, pad]])                      # zero-pad short clips
    audio.set_shape([SAMPLE_RATE])
    return audio, label

N_MELS = 40
mel_matrix = tf.signal.linear_to_mel_weight_matrix(
    num_mel_bins=N_MELS, num_spectrogram_bins=257,
    sample_rate=SAMPLE_RATE, lower_edge_hertz=20.0, upper_edge_hertz=8000.0)

def to_log_mel(audio, label):
    stft = tf.signal.stft(audio, frame_length=400, frame_step=160, fft_length=512)  # 25 ms / 10 ms
    spec = tf.abs(stft)
    mel = tf.tensordot(spec, mel_matrix, 1)
    log_mel = tf.math.log(mel + 1e-6)
    return log_mel[..., tf.newaxis], label                # (frames, mels, 1)

def make_pipeline(ds, training=False):
    ds = ds.map(to_float_fixed_length, num_parallel_calls=AUTOTUNE)
    ds = ds.map(to_log_mel, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()                                       # cache features in RAM after first epoch
    if training:
        ds = ds.shuffle(10_000)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train = make_pipeline(train_ds, training=True)
val = make_pipeline(val_ds)
test = make_pipeline(test_ds)

for x, y in train.take(1):
    print("Batch features:", x.shape, " labels:", y.shape)
INPUT_SHAPE = x.shape[1:]

### Visualise a spectrogram

In [ ]:
a, l = samples[0]
spec, _ = to_log_mel(*to_float_fixed_length(tf.constant(a), l))
plt.figure(figsize=(8, 3))
plt.imshow(tf.transpose(spec[..., 0]), aspect='auto', origin='lower')
plt.title(f"Log-mel: {LABELS[l]}"); plt.xlabel("frame"); plt.ylabel("mel bin")
plt.colorbar(); plt.show()

## 7. Simple CNN baseline

In [ ]:
norm = tf.keras.layers.Normalization()
norm.adapt(train.map(lambda x, y: x).take(200))           # learn mean/std from a subset

model = tf.keras.Sequential([
    tf.keras.Input(shape=INPUT_SHAPE),
    norm,
    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
EPOCHS = 10
history = model.fit(
    train, validation_data=val, epochs=EPOCHS,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
)

plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1); plt.plot(history.history['loss'], label='train'); plt.plot(history.history['val_loss'], label='val'); plt.title('Loss'); plt.legend()
plt.subplot(1, 2, 2); plt.plot(history.history['accuracy'], label='train'); plt.plot(history.history['val_accuracy'], label='val'); plt.title('Accuracy'); plt.legend()
plt.show()

## 8. Evaluate on test set + confusion matrix

In [ ]:
test_loss, test_acc = model.evaluate(test)
print(f"Test accuracy: {test_acc:.4f}")

y_true = np.concatenate([y.numpy() for _, y in test])
y_pred = np.argmax(model.predict(test), axis=1)
cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=NUM_CLASSES).numpy()

plt.figure(figsize=(8, 7))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(NUM_CLASSES), LABELS, rotation=45); plt.yticks(range(NUM_CLASSES), LABELS)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=7,
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix (test)')
plt.tight_layout(); plt.show()

## 9. Save the model

In [ ]:
model.save("speech_commands_cnn.keras")
print("Saved to speech_commands_cnn.keras")

## 10. (Optional) Export raw data to NumPy / WAV files
Useful if you want to use the audio outside TensorFlow (e.g. PyTorch, librosa, MATLAB).

In [ ]:
import wave

EXPORT_DIR = "speech_commands_export"
N_PER_SPLIT = 200          # set to None to export everything (large!)

def export_split(ds, split_name, n=None):
    xs, ys = [], []
    for i, (audio, label) in enumerate(ds if n is None else ds.take(n)):
        a, l = audio.numpy(), int(label.numpy())
        xs.append(np.pad(a[:SAMPLE_RATE], (0, max(0, SAMPLE_RATE - len(a)))))
        ys.append(l)
        folder = os.path.join(EXPORT_DIR, split_name, LABELS[l])
        os.makedirs(folder, exist_ok=True)
        with wave.open(os.path.join(folder, f"{i:06d}.wav"), "wb") as w:
            w.setnchannels(1); w.setsampwidth(2); w.setframerate(SAMPLE_RATE)
            w.writeframes(a.astype(np.int16).tobytes())
    np.savez_compressed(os.path.join(EXPORT_DIR, f"{split_name}.npz"),
                        audio=np.stack(xs), label=np.array(ys), label_names=np.array(LABELS))
    print(f"{split_name}: {len(ys)} clips exported")

for ds, name in [(train_ds, "train"), (val_ds, "validation"), (test_ds, "test")]:
    export_split(ds, name, N_PER_SPLIT)